In [1]:
from __future__ import print_function, absolute_import
import argparse
import os.path as osp
import random
import sys
import time
from datetime import timedelta

import numpy as np
import torch
from torch.utils.data import DataLoader

from pcr import datasets
from pcr.models.bpbreid_encoder import BPBReIDEncoder, BPBReIDModelCfg
from pcr.models.clip_text_encoder import ClipTextEncoder
from pcr.models.prompt_learner import PromptLearner
from pcr.models.relation_blocks import VisualAttentionBlock
from pcr.loss.clip_supcon_loss import SupConLoss
from pcr.utils.config import load_yaml_config
from pcr.utils.data import transforms as T
from pcr.utils.data.preprocessor import Preprocessor
from pcr.utils.logging import Logger
from pcr.utils.lr_scheduler import WarmupCosineLR
from pcr.utils.osutils import mkdir_if_missing
from pcr.utils.visibility_filter import filter_by_visibility


In [2]:
def get_data(name, data_dir):
    return datasets.create(name, osp.join(data_dir, name))


def get_cache_loader(dataset_list, root, height, width, batch_size, workers):
    normalizer = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    transformer = T.Compose([
        T.Resize((height, width), interpolation=3),
        T.ToTensor(),
        normalizer,
    ])
    return DataLoader(
        Preprocessor(dataset_list, root=root, transform=transformer),
        batch_size=batch_size, num_workers=workers, shuffle=False, pin_memory=True)

def cache_part_features(encoder, data_loader):
    """Single full-dataset forward pass under no_grad, caching every image's part embeddings,
    visibility, and real identity label -- mirrors CLIP-ReID's own stage-1 full-dataset feature
    cache, generalized to BPBreID's [M, D] per-branch embeddings."""
    encoder.eval()
    features, visibilities, labels = [], [], []
    with torch.no_grad():
        for imgs, _, pids, _, _ in data_loader:
            f_out, vis = encoder(imgs.cuda())
            features.append(f_out.cpu())
            visibilities.append(vis.cpu())
            labels.append(pids)
    return torch.cat(features, 0), torch.cat(visibilities, 0), torch.cat(labels, 0)


def build_encoder(cfg):
    model_cfg = BPBReIDModelCfg(backbone=cfg.model.backbone)
    model_cfg.masks.parts_num = cfg.model.parts_num
    model_cfg.dim_reduce_output = cfg.model.dim_reduce_output
    encoder = BPBReIDEncoder(model_cfg, checkpoint_path=cfg.model.checkpoint_path or None).cuda()
    encoder.eval()
    for p in encoder.parameters():
        p.requires_grad_(False)
    return encoder

In [3]:
cfg = load_yaml_config("configs/stage1_relational_prompts.yaml")

In [5]:
cfg.data.batch_size = 4

In [6]:
dataset = get_data(cfg.data.dataset, cfg.data.data_dir)
num_identities = dataset.num_train_pids
num_parts = cfg.model.parts_num
num_branches = 1 + num_parts

=> Market1501 loaded
Dataset statistics:
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |   751 |    12936 |         6
  query    |   750 |     3368 |         6
  gallery  |   751 |    15913 |         6
  ----------------------------------------


In [8]:
encoder = build_encoder(cfg)
text_encoder = ClipTextEncoder(clip_arch=cfg.clip.arch, device='cuda').cuda()
prompt_learner = PromptLearner(num_identities, num_parts, text_encoder, n_ctx=cfg.clip.n_ctx,
                                tab_num_heads=cfg.tab.num_heads, tab_num_layers=cfg.tab.num_layers,
                                device='cuda').cuda()
vab = VisualAttentionBlock(dim=cfg.model.dim_reduce_output, num_heads=cfg.vab.num_heads,
                            num_layers=cfg.vab.num_layers).cuda()

=> init weights from normal distribution
Successfully loaded pretrained weights from "examples/logs/stage0_bpa/model_best.pth.tar"


/home/lakshh/workspace/reid/pcr2/pcr/models/relation_blocks.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
/home/lakshh/workspace/reid/pcr2/pcr/models/relation_blocks.py:44: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)


In [11]:
train_set, _ = filter_by_visibility(
    sorted(dataset.train), encoder, cfg.data.height, cfg.data.width,
    cfg.visibility.lambda_v_min, root=dataset.images_dir, batch_size=cfg.data.cache_batch_size,
    workers=cfg.data.workers)
    
cache_loader = get_cache_loader(train_set, dataset.images_dir, cfg.data.height, cfg.data.width,
    cfg.data.cache_batch_size, cfg.data.workers)

==> Visibility filter (threshold=0.50): kept 12824/12936 images, rejected 112


In [ ]:
cached_features, cached_visibility, cached_labels = cache_part_features(encoder, cache_loader)
cached_features = cached_features.cuda()
cached_labels = cached_labels.cuda()
num_images = cached_labels.size(0)
print("==> Cached {} images across {} identities, {} branches".format(
    num_images, num_identities, num_branches))

==> Cached 12824 images across 751 identities, 6 branches


In [13]:
cached_features.shape

torch.Size([12824, 6, 512])

In [15]:
from pcr.loss.clip_infonce_loss import InfoNCELoss

infonce = InfoNCELoss(temperature=cfg.loss.temperature).cuda()
# fg_ctx deliberately excluded -- Algorithm 1 has no foreground term (see module docstring);
# only part_ctx, TextualAttentionBlock (prompt_learner.tab), and VisualAttentionBlock train.
trainable_params = ([prompt_learner.part_ctx] + list(prompt_learner.tab.parameters())
                        + list(vab.parameters()))
optimizer = torch.optim.Adam(trainable_params, lr=cfg.optim.lr,
                                weight_decay=cfg.optim.weight_decay)
scheduler = WarmupCosineLR(optimizer, max_epochs=cfg.optim.epochs,
                            warmup_epochs=cfg.optim.warmup_epochs,
                            warmup_lr_init=cfg.optim.warmup_lr_init,
                            lr_min=cfg.optim.lr_min)
# GradScaler, not raw fp16 backward -- the CLIP text tower runs in fp16 (matches CLIP-ReID's
# own dtype exactly, see pcr/models/clip_text_encoder.py's docstring), and CLIP-ReID's own
# stage-1 loop always wraps its backward in a GradScaler to guard against fp16 gradient
# underflow through the text transformer -- ported faithfully rather than assuming raw fp16
# backward is fine. VisualAttentionBlock and PromptLearner's own parameters run in fp32
# (GradScaler is harmless for fp32 leaves), so one scaler covers everything trainable.
scaler = torch.amp.GradScaler('cuda')

In [16]:
def build_pk_batches(cached_labels, num_instances, batch_size):
    """Groups the cached feature set's indices by identity, then partitions all identities into
    PK batches for one epoch: batch_size // num_instances identities per batch, num_instances
    cached images per identity (sampled with replacement if that identity has fewer than
    num_instances cached images). Algorithm 1 step 6 ("Sample a PK batch of pre-filtered
    images") -- see this file's own module docstring for why this matters even with InfoNCELoss's
    per-identity deduplication making it safe against PK-batch collisions. A final partial group
    of identities (fewer than batch_size // num_instances left over) is dropped, matching this
    repo's other PK
    samplers' drop_last convention (pcr/utils/data/sampler.py::RandomIdentitySampler)."""
    labels_np = cached_labels.cpu().numpy()
    id_to_indices = {}
    for idx, pid in enumerate(labels_np):
        id_to_indices.setdefault(int(pid), []).append(idx)
    pids = list(id_to_indices.keys())
    random.shuffle(pids)

    num_pids_per_batch = max(1, batch_size // num_instances)
    batches = []
    for start in range(0, len(pids), num_pids_per_batch):
        batch_pids = pids[start:start + num_pids_per_batch]
        if len(batch_pids) < num_pids_per_batch:
            break
        batch_idx = []
        for pid in batch_pids:
            pool = id_to_indices[pid]
            replace = len(pool) < num_instances
            chosen = np.random.choice(pool, size=num_instances, replace=replace)
            batch_idx.extend(int(i) for i in chosen)
        batches.append(torch.tensor(batch_idx, dtype=torch.long, device=cached_labels.device))
    return batches

prompt_learner.train()
vab.train()
epoch_loss = 0.0
epoch_start = time.time()
# Algorithm 1 step 6: a fresh PK partition of the cached feature set every epoch, not a
# plain random sub-batch -- see build_pk_batches' and this file's own module docstring.
batches = build_pk_batches(cached_labels, cfg.data.num_instances, cfg.data.batch_size)
iters_per_epoch = len(batches)

In [17]:
b_idx = batches[0]
b_labels = cached_labels[b_idx]
b_features = cached_features[b_idx]  # [b, 1+K, D], already L2-normalized per branch

optimizer.zero_grad()

# Algorithm 1 steps 10-14: only the K part prompts are built and pushed through the
# frozen CLIP text encoder -- no foreground prompt/loss (module docstring).
prompts = prompt_learner.build_part_prompts(b_labels)  # list of 1+K tensors
part_visual = vab(b_features[:, 1:, :])  # [b, K, D], relationally mixed
loss = b_features.new_zeros(())

In [21]:
len(prompts[0])
prompts[0].shape

torch.Size([4, 77, 512])

In [22]:
part_visual = vab(b_features[:, 1:, :])

In [ ]:
part_visual[0].shape

torch.Size([5, 512])

: 